In [1]:
import os        # 작업 폴더 변경용
import pathlib   # 경로를 객체로 다루기
import sys       # 모듈 검색 경로(sys.path) 조작용

here = pathlib.Path.cwd()                                   # 현재 작업 폴더
ROOT = here.parents[2] if here.name == "Wed" else here      # Wed에서 실행했으면 3단계 위가 프로젝트 루트
os.chdir(ROOT)                                              # 작업 폴더를 루트로 이동
SANDBOX = ROOT / "sandbox" / "W5" / "Wed"                   # 오늘 실습 폴더
SAMPLES = ROOT / "sandbox" / "W5" / "Mon" / "samples"       # 샘플 문서(재료) 폴더

print("프로젝트 루트  :", ROOT)       # 루트 경로 확인
print("재료 폴더      :", SAMPLES)    # 샘플 폴더 경로 확인

if str(ROOT / 'backend') not in sys.path:          # backend가 import 경로에 없으면
    sys.path.insert(0, str(ROOT / 'backend'))      # 맨 앞에 추가 → `from app...` import 가능

프로젝트 루트  : d:\hanwha-agent
재료 폴더      : d:\hanwha-agent\sandbox\W5\Mon\samples


In [2]:
# 말 한 마디를 대화 이력에 쌓는 함수
def append_turn(history:list[dict[str,str]], role:str, content:str)->None:
    history.append({'role':role, 'content':content})   # {역할, 내용} 딕셔너리를 이력 끝에 추가

chat: list[dict[str,str]] = []                                         # 빈 대화 이력
append_turn(chat, 'user', 'How is the weather today?')                 # 사용자 발화 추가
append_turn(chat, 'assistant', 'No access to the forecast database')   # 어시스턴트 발화 추가

for message in chat:                                        # 이력을 순서대로 돌면서
    print(f"{message['role']:<10}{message['content']}")     # 역할(10칸 왼쪽 정렬) + 내용 출력

print(f'대화 이력 길이:{len(chat)}개')                        # 쌓인 발화 수 출력

user      How is the weather today?
assistant No access to the forecast database
대화 이력 길이:2개


In [3]:
if not SAMPLES.is_dir():                     # 샘플 폴더가 없으면
    print('재료 폴더가 없습니다')              # 안내만 출력
else:
    all_files = [p for p in SAMPLES.rglob('*') if p.is_file()]                  # 하위 폴더까지 모든 파일
    top_files = [p for p in SAMPLES.iterdir() if p.is_file()]                   # samples 바로 아래 파일만
    sub_files = [p for p in (SAMPLES / '_formats').rglob('*') if p.is_file()]   # _formats 폴더 안 파일
    today = sorted((p.name for p in SAMPLES.glob('DOC-HR-014_*_v2.0.*')))       # 오늘 쓸 v2.0 파일 이름들(정렬)
    print(f'재료 파일 : {len(all_files)}개')                                      # 전체 파일 수
    print(f'  samples 바로 아래 : {len(top_files)}개 · _formats 안 : {len(sub_files)}개')  # 위치별 파일 수
    print('오늘 읽을 세 형식 :')
    for name in today:          # v2.0 파일들을
        print(' ', name)        # 한 줄씩 출력

재료 파일 : 30개
  samples 바로 아래 : 24개 · _formats 안 : 6개
오늘 읽을 세 형식 :
  DOC-HR-014_국내출장_여비_규정_v2.0.docx
  DOC-HR-014_국내출장_여비_규정_v2.0.hwpx
  DOC-HR-014_국내출장_여비_규정_v2.0.pdf


In [4]:
from app.rag.local_parsers import parse_local                 # 파일 형식별 로컬 파서
TARGET = next(SAMPLES.glob('DOC-HR-014_*_v2.0.docx'))         # v2.0 docx 파일 하나 찾기
doc = parse_local(TARGET)                                     # 파싱 → 블록 목록을 가진 문서 객체
print(f'블록 {len(doc.blocks)} · 표 {doc.table_count} · 쪽 {doc.page_count}')   # 블록/표/쪽 수 요약

블록 233 · 표 7 · 쪽 1


In [5]:
for i, b in enumerate(doc.blocks[:5]):                           # 앞 5개 블록만
    head = b.text[:36] + ('…' if len(b.text) > 36 else '')       # 본문 앞 36자, 넘치면 … 붙이기
    print(f'[{i:2d}] {b.kind} | {b.locator} | {head}')           # 번호 | 종류 | 위치 | 미리보기
first_table = next((b for b in doc.blocks if b.kind == '표'))    # 종류가 '표'인 첫 블록
print()
print('첫 표 블록 : locator=' + first_table.locator)             # 표 위치(예: 표1)
print(' ', first_table.text[:60])                               # 표 내용(마크다운) 앞 60자

[ 0] 조항 | DOC-HR-014_국내출장_여비_규정_v2.0 | 일반
[ 1] 조항 | DOC-HR-014_국내출장_여비_규정_v2.0 | 국내출장 여비 규정
[ 2] 조항 | DOC-HR-014_국내출장_여비_규정_v2.0 | v2.0
[ 3] 조항 | DOC-HR-014_국내출장_여비_규정_v2.0 | 문서번호  DOC-HR-014
[ 4] 조항 | DOC-HR-014_국내출장_여비_규정_v2.0 | 시행일    2025-07-01

첫 표 블록 : locator=표1
  | 버전 | 구분 | 시행일 | 주요 내용 | 작성 | 검토 | 승인 |
|---|---|---|---|--


In [6]:
# 두 문자열의 글자 집합 유사도(자카드) 계산
def jaccard(x: str, y: str) -> float:
    sx, sy = (set(x.replace(' ', '')), set(y.replace(' ', '')))   # 공백 빼고 글자 집합으로
    return len(sx & sy) / len(sx | sy)                            # 교집합 크기 / 합집합 크기
PAIRS = [('광역시 숙박비 상한', '잠자리 비용 한도'), ('광역시 숙박비 상한', '광역시 숙박비 상한액')]   # 비교할 문장 쌍
for left, right in PAIRS:                                                        # 쌍마다
    shared = sorted(set(left.replace(' ', '')) & set(right.replace(' ', '')))   # 겹치는 글자들
    shown = shared if len(shared) <= 3 else f'{len(shared)}개'                  # 3개 이하면 글자, 많으면 개수만
    print(f"'{left}' vs '{right}'  겹치는 글자 {shown} · 자카드 {jaccard(left, right):.3f}")   # 결과 출력

'광역시 숙박비 상한' vs '잠자리 비용 한도'  겹치는 글자 ['비', '한'] · 자카드 0.154
'광역시 숙박비 상한' vs '광역시 숙박비 상한액'  겹치는 글자 8개 · 자카드 0.889


In [7]:
from app.rag.local_parsers import parse_local           # 로컬 파서

DOCX = next(SAMPLES.glob('DOC-HR-014_*_v2.0.docx'))     # v2.0 docx 파일 찾기
doc = parse_local(DOCX)                                 # 블록 단위로 파싱

print('연 문서 :', DOCX.name, '· 블록', len(doc.blocks), '개')   # 파일명과 블록 수
print()

# SQL LIKE '%word%'처럼 단어가 들어간 블록 수 세기
def like(word: str) -> int:
    return sum((1 for b in doc.blocks if word in b.text))   # 포함된 블록마다 1씩 더함

for word in ['숙박비', '잠자리', '잠자리 비용']:       # 같은 뜻, 다른 표현으로 검색
    print(f"LIKE '%{word}%' → {like(word)}건")       # 글자가 정확히 일치해야만 찾음을 확인


연 문서 : DOC-HR-014_국내출장_여비_규정_v2.0.docx · 블록 233 개

LIKE '%숙박비%' → 27건
LIKE '%잠자리%' → 0건
LIKE '%잠자리 비용%' → 0건


In [8]:
import math                                    # 제곱근 계산용
# 문장을 2차원 좌표(임베딩 흉내)로 표현
POINTS = {'광역시 숙박비 상한': [0.9, 0.2], '잠자리 비용 한도': [0.86, 0.28], '출장 신청 절차': [0.1, 0.95]}

# 두 좌표 사이의 유클리드 거리
def distance(p: list[float], q: list[float]) -> float:
    return math.sqrt(sum(((a - b) ** 2 for a, b in zip(p, q))))   # 좌표별 차이 제곱합의 제곱근

BASE = '광역시 숙박비 상한'                      # 기준 문장
print('기준 :', BASE, POINTS[BASE])

for text, point in POINTS.items():             # 모든 문장에 대해
    if text == BASE:                           # 기준 문장 자신은
        continue                               # 건너뛰기
    gap = distance(POINTS[BASE], point)        # 기준과의 거리
    print(f"  {text}  {point} → 거리 {gap:.3f}   {('← 가깝다' if gap < 0.5 else '← 멀다')}")   # 0.5 미만이면 의미가 가깝다


기준 : 광역시 숙박비 상한 [0.9, 0.2]
  잠자리 비용 한도  [0.86, 0.28] → 거리 0.089   ← 가깝다
  출장 신청 절차  [0.1, 0.95] → 거리 1.097   ← 멀다


In [9]:
v2 = [0.9, 0.2]                                         # 2차원 벡터
v3 = [0.6, 0.8, 0.0]                                    # 3차원 벡터
v1024 = [round(math.sin(i), 4) for i in range(1024)]    # 1024차원 벡터(sin 값으로 채운 가짜 임베딩)


# 벡터의 크기(길이, ‖v‖) 계산
def size(v: list[float]) -> float:
    return math.sqrt(sum(x * x for x in v))             # 각 성분 제곱합의 제곱근


for label, v in [("2차원", v2), ("3차원", v3), ("1024차원", v1024)]:   # 세 벡터를 차례로
    # 문법 - v[:5] 는 앞에서 다섯 개만 잘라 낸 새 리스트다(슬라이싱)
    print(f"{label:<8} : 길이 {len(v):<4} 앞 5개 {v[:5]}")          # 이름 | 차원 수 | 앞 5개 값

print()
print(f"크기(‖v‖) : {size(v2):.4f} · {size(v3):.4f} · {size(v1024):.4f}")   # 세 벡터의 크기 비교(v3는 크기 1)

2차원      : 길이 2    앞 5개 [0.9, 0.2]
3차원      : 길이 3    앞 5개 [0.6, 0.8, 0.0]
1024차원   : 길이 1024 앞 5개 [0.0, 0.8415, 0.9093, 0.1411, -0.7568]

크기(‖v‖) : 0.9220 · 1.0000 · 22.6283


In [10]:
# 내적 
def dot(u: list[float], v: list[float]) -> float:
    return sum((a * b for a, b in zip(u, v)))

# 크기 
def norm(u: list[float]) -> float:
    return math.sqrt(dot(u, u))

# 코사인 유사도  
def cosine(u: list[float], v: list[float]) -> float:
    return dot(u, v) / (norm(u) * norm(v))

u = [0.6, 0.8, 0]
v = [0, 1, 0]

print('u =', u, '  v =', v)
print(f'  내적   : {dot(u, v):+.4f}')
print(f'  크기   : {norm(u):.4f} × {norm(v):.4f}')
print(f'  코사인 : {cosine(u, v):+.4f}')


u = [0.6, 0.8, 0]   v = [0, 1, 0]
  내적   : +0.8000
  크기   : 1.0000 × 1.0000
  코사인 : +0.8000


In [11]:
w = [-0.6, -0.8, 0]    # u 를 정확히 뒤집은 벡터
perp = [0, 0, 1]       # u 와 직각인 벡터 (perpendicular)

CASES = [
    ("u", u,    "똑같다"),
    ("v", v,    "닮았다"),
    ("⊥", perp, "아무 상관 없다"),
    ("w", w,    "정반대"),
]
for label, other, meaning in CASES:
    print(f"cos(u, {label}) = {cosine(u, other):+.4f}    {meaning}")

cos(u, u) = +1.0000    똑같다
cos(u, v) = +0.8000    닮았다
cos(u, ⊥) = +0.0000    아무 상관 없다
cos(u, w) = -1.0000    정반대


In [12]:
def unit(v: list[float]) -> list[float]:
    n = norm(v)
    return [x / n for x in v]

u = [0.6, 0.8, 0]
v = [4.0, 3.0, 0]
uh, vh = (unit(u), unit(v))
print(f'‖û‖ = {norm(uh):.1f}   ‖v̂‖ = {norm(vh):.1f}')

gap = abs(dot(uh, vh) - cosine(u, v))
print(f'dot(û, v̂) = {dot(uh, vh):.6f}   cosine(u, v) = {cosine(u, v):.6f}   차이 {gap:.1f}')
print('차이가 1e-12 보다 작은가 :', gap < 1e-12)

‖û‖ = 1.0   ‖v̂‖ = 1.0
dot(û, v̂) = 0.960000   cosine(u, v) = 0.960000   차이 0.0
차이가 1e-12 보다 작은가 : True


In [13]:
BYTES_PER_NUMBER = 4

def mb(dim: int, n_chunks: int) -> float:
    return dim * n_chunks * BYTES_PER_NUMBER / 1024 / 1024

CASES = [(1024, 70, ''), (1024, 8000, ''), (4096, 70, ''), (4096, 8000, '     ← 4배')]
for dim, n, note in CASES:
    print(f'{dim}차원 × {n:>4}청크 × 4바이트 = {mb(dim, n):>6.2f} MB{note}')
print()
print(f'4096차원은 1024차원의 {mb(4096, 8000) / mb(1024, 8000):.0f}배다')


1024차원 ×   70청크 × 4바이트 =   0.27 MB
1024차원 × 8000청크 × 4바이트 =  31.25 MB
4096차원 ×   70청크 × 4바이트 =   1.09 MB
4096차원 × 8000청크 × 4바이트 = 125.00 MB     ← 4배

4096차원은 1024차원의 4배다


In [14]:
MODELS = ROOT.parent / 'models'      # download_models.py가 받는 자리: hanwha-agent 바로 옆 models/
KURE = MODELS / 'KURE-v1'            # 한국어 특화 임베딩 모델
BGE = MODELS / 'bge-m3'              # 다국어 임베딩 모델
MODELS_READY = KURE.is_dir() and BGE.is_dir()   # 두 폴더가 다 있어야 True
print('KURE-v1 :', KURE)
print('bge-m3  :', BGE)
print('MODELS_READY :', MODELS_READY)

KURE-v1 : d:\models\KURE-v1
bge-m3  : d:\models\bge-m3
MODELS_READY : True


In [15]:
def show_folder(title, folder):

    files = sorted((p for p in folder.iterdir() if p.is_file()), key=lambda p: p.stat().st_size, reverse=True)
    total = sum((p.stat().st_size for p in folder.rglob('*') if p.is_file()))
    print(f'{title}  (약 {total / 1000000000.0:.2f} GB)')
    for p in files[:5]:
        print(f'  {p.name:<24} {p.stat().st_size / 1000000.0:>8.1f} MB')
if MODELS_READY:
    show_folder('KURE-v1', KURE)
    show_folder('bge-m3 ', BGE)
else:
    print('모델 폴더가 없어 건너뜁니다. 오늘 진도는 그대로 나갑니다.')


KURE-v1  (약 2.29 GB)
  model.safetensors          2271.1 MB
  tokenizer.json               17.1 MB
  tokenizer_config.json         0.0 MB
  special_tokens_map.json       0.0 MB
  config.json                   0.0 MB
bge-m3   (약 2.29 GB)
  pytorch_model.bin          2271.1 MB
  tokenizer.json               17.1 MB
  sentencepiece.bpe.model       5.1 MB
  special_tokens_map.json       0.0 MB
  config.json                   0.0 MB


In [16]:
PAIRS = [
    ("광역시 숙박비 상한", "잠자리 비용 한도", "같은 뜻"),
    ("출장 전에 신청서를 낸다", "출장은 사전 신청이 원칙이다", "같은 뜻"),
    ("법인카드로 결제한다", "법인카드 사용 지침을 따른다", "가까움"),
    ("일비는 하루 단위로 준다", "숙박비는 1박 단위로 준다", "가까움"),
    ("출장 신청서를 낸다", "재택근무를 신청한다", "다름"),
    ("숙박비 상한액", "정보보안 지침 위반", "남남"),
]

CLOSE = {"같은 뜻", "가까움"}

near = sum(1 for a, b, label in PAIRS if label in CLOSE)
print(f"문장쌍 {len(PAIRS)}개 (가까운 쌍 {near} · 먼 쌍 {len(PAIRS) - near})")

for i, (a, b, label) in enumerate(PAIRS, start=1):
    print(f" {i} [{label}] {a}  ↔  {b}")

문장쌍 6개 (가까운 쌍 4 · 먼 쌍 2)
 1 [같은 뜻] 광역시 숙박비 상한  ↔  잠자리 비용 한도
 2 [같은 뜻] 출장 전에 신청서를 낸다  ↔  출장은 사전 신청이 원칙이다
 3 [가까움] 법인카드로 결제한다  ↔  법인카드 사용 지침을 따른다
 4 [가까움] 일비는 하루 단위로 준다  ↔  숙박비는 1박 단위로 준다
 5 [다름] 출장 신청서를 낸다  ↔  재택근무를 신청한다
 6 [남남] 숙박비 상한액  ↔  정보보안 지침 위반


In [17]:
def jaccard(x, y):
    sx, sy = (set(x.replace(' ', '')), set(y.replace(' ', '')))
    return len(sx & sy) / len(sx | sy)
scores = {True: [], False: []}
for i, (a, b, label) in enumerate(PAIRS, start=1):
    value = jaccard(a, b)
    scores[label in CLOSE].append(value)
    print(f' {i} [{label}] 자카드 {value:.3f}')
print()
print(f'가까운 쌍 평균 : {sum(scores[True]) / len(scores[True]):.3f}')
print(f'먼 쌍 평균     : {sum(scores[False]) / len(scores[False]):.3f}')

 1 [같은 뜻] 자카드 0.154
 2 [같은 뜻] 자카드 0.400
 3 [가까움] 자카드 0.312
 4 [가까움] 자카드 0.538
 5 [다름] 자카드 0.308
 6 [남남] 자카드 0.000

가까운 쌍 평균 : 0.351
먼 쌍 평균     : 0.154


In [18]:
from app.rag.local_parsers import parse_local

docx = next(SAMPLES.glob('DOC-HR-014_*_v2.0.docx'))
doc = parse_local(docx)

tables = [b for b in doc.blocks if b.kind == '표']
print(f'표 블록 {len(tables)}개')

for b in tables:
    print(f'  {b.locator}  {len(b.text)}자')
print()
print('표4 앞 6줄 :')
for line in tables[3].text.split('\n')[:6]:
    print(line)

표 블록 7개
  표1  291자
  표2  242자
  표3  210자
  표4  378자
  표5  324자
  표6  302자
  표7  296자

표4 앞 6줄 :
| 교통수단 | 등급 | 지급 기준 | 승인 | 비고 |
|---|---|---|---|---|
| 철도(KTX·SRT) | 일반실 | 실비 | 팀장 | 3급 이상은 특실 가능 |
| 일반철도(무궁화·새마을) | 일반실 | 실비 | 팀장 |  |
| 고속·시외버스 | 우등 이하 | 실비 | 팀장 |  |
| 국내선 항공 | 일반석 | 실비 | 본부장 | 제19조 요건 충족 시 |


In [ ]:
joined = '\n'.join((b.text for b in doc.blocks)) # 문서의 모든 텍스트 모으기
naive = [joined[i:i + 1200] for i in range(0, len(joined), 1200)]
print(f'{len(joined):,}자 → {len(naive)} 청크')

for b in tables:
    start = joined.find(b.text)
    end = start + len(b.text)
    first, last = (start // 1200, (end - 1) // 1200)
    mark = '  두 조각으로 갈린다' if first != last else '' # 표가 왜 잘리는 거지?
    print(f'  {b.locator}: 원문 위치 {start:4d}~{end:4d} → 청크 #{first}~#{last}{mark}')


8,505자 → 8 청크
  표1: 원문 위치  115~ 406 → 청크 #0~#0
  표2: 원문 위치 6370~6612 → 청크 #5~#5
  표3: 원문 위치 6701~6911 → 청크 #5~#5
  표4: 원문 위치 7035~7413 → 청크 #5~#6  두 조각으로 갈린다
  표5: 원문 위치 7475~7799 → 청크 #6~#6
  표6: 원문 위치 7833~8135 → 청크 #6~#6
  표7: 원문 위치 8191~8487 → 청크 #6~#7  두 조각으로 갈린다


In [20]:
table4 = tables[3]
start = joined.find(table4.text)
end = start + len(table4.text)
cut = 1200 * ((end - 1) // 1200)
print('── 앞 조각 끝부분 ──')
print(joined[start:cut])
print()
print('── 뒤 조각 앞부분 ──')
print(joined[cut:end])
print()
header = '| 교통수단 |'
print('앞 조각에 머리행이 있나 :', header in joined[start:cut])
print('뒤 조각에 머리행이 있나 :', header in joined[cut:end])


── 앞 조각 끝부분 ──
| 교통수단 | 등급 | 지급 기준 | 승인 | 비고 |
|---|---|---|---|---|
| 철도(KTX·SRT) | 일반실 | 실비 | 팀장 | 3급 이상은 특실 가능 |
| 일반철도(무궁화·새마을) | 일반실 | 실비 | 팀장 |  |
| 고속·시외버스 | 우등 이하 | 실비 | 팀장

── 뒤 조각 앞부분 ──
 |  |
| 국내선 항공 | 일반석 | 실비 | 본부장 | 제19조 요건 충족 시 |
| 선박 | 2등실 이상 | 실비 | 팀장 | 도서 지역 출장 |
| 자가용 | — | 1km당 300원 | 팀장(사전) | 유류비·감가상각 포함 |
| 택시 | — | 실비 | 팀장 | 심야·대중교통 미운행 시에 한함 |
| 렌터카 | 중형 이하 | 실비 | 본부장 | 2인 이상 동행 시 |

앞 조각에 머리행이 있나 : True
뒤 조각에 머리행이 있나 : False


In [28]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=0)
pieces = splitter.split_text(joined)

print(f'{len(joined):,}자 → {len(pieces)} 청크')
for b in tables:
    whole = any((b.text in p for p in pieces))
    print(f"  {b.locator}: {('한 조각에 통째로' if whole else '★ 두 조각으로 갈린다')}")
print('조각 하나의 모양 :', type(pieces[0]).__name__)


8,505자 → 5 청크
  표1: 한 조각에 통째로
  표2: 한 조각에 통째로
  표3: 한 조각에 통째로
  표4: 한 조각에 통째로
  표5: 한 조각에 통째로
  표6: ★ 두 조각으로 갈린다
  표7: 한 조각에 통째로
조각 하나의 모양 : str


In [30]:
LIMIT = 1200

for b in tables:
    print(f"{b.locator}  {len(b.text)}자  넘나? {len(b.text) > LIMIT}")

print()

longest = max(len(b.text) for b in tables)
over = sum(1 for b in tables if len(b.text) > LIMIT)   # 조건에 맞는 것의 개수를 센다
print(f"가장 긴 표 = {longest}자 · LIMIT 을 넘는 표 = {over}개")

표1  291자  넘나? False
표2  242자  넘나? False
표3  210자  넘나? False
표4  378자  넘나? False
표5  324자  넘나? False
표6  302자  넘나? False
표7  296자  넘나? False

가장 긴 표 = 378자 · LIMIT 을 넘는 표 = 0개


In [31]:
import re

text = '제1조(목적) 이 규정은 여비를 정한다. 제8조(일비) 일비는 30,000원으로 한다.'

print('캡처 그룹 ()      ← 우리가 쓸 형태')
print(re.split('(제\\s*\\d+\\s*조(?:의\\s*\\d+)?\\s*(?:\\([^)]{1,30}\\))?)', text))
print('  parts[0] = 머리 · parts[1],[3],… = 조 제목 · parts[2],[4],… = 본문   (홀짝 규칙)')
print()

print('비캡처 (?:)      ← 조 제목이 통째로 사라진다')
print(re.split('(?:제\\s*\\d+\\s*조(?:의\\s*\\d+)?\\s*(?:\\([^)]{1,30}\\))?)', text))
print()

print('안쪽까지 캡처     ← 그룹이 셋이라 홀짝 규칙이 깨진다')
print(re.split('(제\\s*\\d+\\s*조(의\\s*\\d+)?\\s*(\\([^)]{1,30}\\))?)', text))


캡처 그룹 ()      ← 우리가 쓸 형태
['', '제1조(목적)', ' 이 규정은 여비를 정한다. ', '제8조(일비)', ' 일비는 30,000원으로 한다.']
  parts[0] = 머리 · parts[1],[3],… = 조 제목 · parts[2],[4],… = 본문   (홀짝 규칙)

비캡처 (?:)      ← 조 제목이 통째로 사라진다
['', ' 이 규정은 여비를 정한다. ', ' 일비는 30,000원으로 한다.']

안쪽까지 캡처     ← 그룹이 셋이라 홀짝 규칙이 깨진다
['', '제1조(목적)', None, '(목적)', ' 이 규정은 여비를 정한다. ', '제8조(일비)', None, '(일비)', ' 일비는 30,000원으로 한다.']


In [32]:
import importlib
from app.rag import chunker

importlib.reload(chunker)

body = '\n'.join((b.text for b in doc.blocks if b.kind == '조항'))

parts = chunker._split_articles(body)

print(f'조각 {len(parts)}개')

for title, whole in parts[:5]:
    head = whole[:30] + (' …' if len(whole) > 30 else '')
    print(f'  ({title!r}, {head!r})')

조각 101개
  ('', '일반\n국내출장 여비 규정\nv2.0\n문서번호  DOC-H …')
  ('제9조', '제9조에 따라 관리한다.\n목차\n제1장 총칙')
  ('제1조(목적)', '제1조(목적)')
  ('제2조(정의)', '제2조(정의)')
  ('제3조(적용 범위)', '제3조(적용 범위)')


In [34]:
importlib.reload(chunker)
long_body = '제7조(일비) ' + '출장자는 출장 기간 중 일비를 지급받는다. ' * 60
pieces = chunker._hard_wrap(long_body)
print(f'합성 예제 {len(long_body):,}자 → {len(pieces)}조각')
for i, piece in enumerate(pieces):
    print(f'  [{i}] {len(piece):4d}자')
print()
real_long = [whole for _, whole in chunker._split_articles(body) if len(whole) > 1200]
print(f'실문서에서 1,200자를 넘는 조항 : {len(real_long)}개')


합성 예제 1,448자 → 2조각
  [0] 1183자
  [1]  263자

실문서에서 1,200자를 넘는 조항 : 0개


In [35]:
importlib.reload(chunker)

from app.rag.chunker import chunk, summarize

chunks = chunk(doc)
print(summarize(chunks))
print()
for c in chunks[:5]:
    print(f'  {c.kind:<4s} | {c.locator:<34s} | {len(c.text):3d}자')
print()
print('표 청크 locator :', [c.locator for c in chunks if c.kind == '표'])
sizes = [len(c.text) for c in chunks]
print(f'청크 글자 수    : 최대 {max(sizes)} · 최소 {min(sizes)} · 평균 {sum(sizes) / len(sizes):.1f}')


조항 단위 63 + 표 단위 7 = 70 청크

  조항   | DOC-HR-014_국내출장_여비_규정_v2.0         | 114자
  표    | 표1                                 | 291자
  조항   | 제9조                                |  66자
  조항   | 제5조(여비의 종류)                        |  74자
  조항   | 제10조(출장의 취소)                       | 239자

표 청크 locator : ['표1', '표2', '표3', '표4', '표5', '표6', '표7']
청크 글자 수    : 최대 378 · 최소 10 · 평균 120.2


In [37]:
before = chunker.TOC_BODY_MIN
chunker.TOC_BODY_MIN = 0
off = chunker.chunk(doc)
chunker.TOC_BODY_MIN = before

def report(label, cs):
    """두 결과를 같은 폭으로 나란히 찍는다."""
    articles = sum((1 for c in cs if c.kind == '조항'))
    tables_n = sum((1 for c in cs if c.kind == '표'))
    avg = sum((len(c.text) for c in cs)) / len(cs)
    return f'{label} : 조항 단위 {articles:3d} + 표 단위 {tables_n} = {len(cs):3d} 청크 · 평균 {avg:5.1f}자'
print(report('정상 (TOC_BODY_MIN = 10)', chunks))
print(report('끔   (TOC_BODY_MIN =  0)', off))
print(f'차이 = {len(off) - len(chunks)}개')
print()
print(f'30자 미만 청크 = {sum((1 for c in off if len(c.text) < 30))}개 · 가장 짧은 것들 —')
for c in sorted(off, key=lambda c: len(c.text))[:6]:
    tag = f'{c.locator!r},'
    print(f'  ({tag:<16s}{len(c.text)}자)')


정상 (TOC_BODY_MIN = 10) : 조항 단위  63 + 표 단위 7 =  70 청크 · 평균 120.2자
끔   (TOC_BODY_MIN =  0) : 조항 단위 106 + 표 단위 7 = 113 청크 · 평균  74.1자
차이 = 43개

30자 미만 청크 = 53개 · 가장 짧은 것들 —
  ('제1조(목적)',      7자)
  ('제2조(정의)',      7자)
  ('제5조',          7자)
  ('제5조',          7자)
  ('제11조(일비)',     8자)
  ('제12조(식비)',     8자)


In [38]:
import re

bare = [c for c in chunks if c.kind == "조항" and re.fullmatch(r"제\d+조", c.locator)]
print(f"① 괄호 제목 없는 「제N조」 locator : {len(bare)}개 — 그중 본문 조각 넷")
for c in bare[4:8]:
    print(f"   {c.locator:5s} ← {c.text[:30]!r}")

mixed = [c for c in chunks if re.search(r"\n\s*제\d+장", c.text)]
print(f"② 본문에 장 제목 줄이 섞인 청크 : {len(mixed)}개 — 예: {mixed[2].locator}")
print(f"   … {mixed[2].text[-22:]!r}")

print(f"③ 「제N장」 으로 시작하는 locator : {sum(bool(re.match(r'제\d+장', c.locator)) for c in chunks)}개")

① 괄호 제목 없는 「제N조」 locator : 11개 — 그중 본문 조각 넷
   제32조  ← '제32조에 따라 반환한다.\n③ 회사의 사정으로 출장이 '
   제8조   ← '제8조제2항에 따라 본부장의 승인을 받아야 한다.\n④ '
   제7조   ← '제7조제1항의 신청 기한을 적용하지 아니한다.\n② 긴급'
   제14조  ← '제14조의 상한을 초과하여 지급할 수 있다. 이 경우 '
② 본문에 장 제목 줄이 섞인 청크 : 9개 — 예: 제10조(출장의 취소)
   … '제24조(교육 및 연수)\n제25조(파견)'
③ 「제N장」 으로 시작하는 locator : 0개


In [40]:
importlib.reload(chunker)
from app.rag.chunker import chunk, summarize   

chunks = chunk(doc)            
print(summarize(chunks))

bare = [c for c in chunks if c.kind == "조항" and re.fullmatch(r"제\d+조", c.locator)]
mixed = [c for c in chunks if re.search(r"\n\s*제\d+장", c.text)]
print(f"괄호 제목 없는 「제N조」 locator : {len(bare)}개")
print(f"본문에 장 제목 줄이 섞인 청크 : {len(mixed)}개")
print(f"「제N장」 으로 시작하는 locator : {sum(bool(re.match(r'제\d+장', c.locator)) for c in chunks)}개")
print()

for c in chunks:
    if c.locator.startswith(("제3장 제2절 제14조", "제4장 제26조", "부칙 제1조")):
        print(f"   {c.locator:24s} … {c.text[-22:]!r}")


조항 단위 46 + 표 단위 7 = 53 청크
괄호 제목 없는 「제N조」 locator : 0개
본문에 장 제목 줄이 섞인 청크 : 0개
「제N장」 으로 시작하는 locator : 35개

   제3장 제2절 제14조(숙박비) · 별표1  … '영수증을 첨부한 경우에 한하여 지급한다.'
   제4장 제26조(재해 대응 출장)       … '경우 복귀 후 사유서를 제출하여야 한다.'
   부칙 제1조(시행일)              … '정은 2025년 7월 1일부터 시행한다.'


In [41]:
path = ROOT / "backend" / "tests" / "test_chunker.py"
text = path.read_text(encoding="utf-8")
print(path.name, "·", len(text.splitlines()), "줄")
print("  테스트 함수 :", text.count("\ndef test_"))

test_chunker.py · 54 줄
  테스트 함수 : 4


In [42]:
from app.rag.local_parsers import parse_local
from app.rag.chunker import chunk, summarize
parsed, chunks = ({}, {})
for ext in ('hwpx', 'docx', 'pdf'):
    path = next(SAMPLES.glob(f'DOC-HR-014_*_v2.0.{ext}'))
    doc = parse_local(path)
    cs = chunk(doc)
    parsed[ext], chunks[ext] = (doc, cs)
    print(f'[{ext:4}] blocks={len(doc.blocks):3} tables={doc.table_count} pages={doc.page_count:2} → {summarize(cs)}')

[hwpx] blocks=233 tables=7 pages= 1 → 조항 단위 46 + 표 단위 7 = 53 청크
[docx] blocks=233 tables=7 pages= 1 → 조항 단위 46 + 표 단위 7 = 53 청크
[pdf ] blocks= 16 tables=0 pages=16 → 조항 단위 45 + 표 단위 0 = 45 청크


In [ ]:
for ext in ('docx', 'pdf'):
    doc = parsed[ext]
    kinds = sorted({b.kind for b in doc.blocks})
    bars = '\n'.join((b.text for b in doc.blocks)).count('|')
    print(f'{ext:4} 블록 종류 : {str(kinds):16} 마크다운 표 기호 | : {bars}개')
print()
target = next((c for c in chunks['pdf'] if '숙박비(광역시)' in c.text))
print(f'pdf 쪽 청크 : kind={target.kind} · locator={target.locator} · {len(target.text)}자')
print('  ' + target.text[:80].replace('\n', ' ⏎ ') + ' …')

docx 블록 종류 : ['조항', '표']      마크다운 표 기호 | : 323개
pdf  블록 종류 : ['조항']           마크다운 표 기호 | : 0개

pdf 쪽 청크 : kind=조항 · locator=별표1 · 281자
  [별표1] 직급별 여비 지급 기준표 (제5조제2항, 제11조제3항, 제12조제2항, 제14조제1항 관련) ⏎ 직급 ⏎ 일비 ⏎ 숙박비(광역시) ⏎ 숙박비(그  …
